============================================================
Notebook 13 — ABIDE multi-site comparison (site selection)

Purpose: justify choosing a SINGLE ABIDE site (NYU) for the near-Gaussian
real-data leg, by measuring per-site sample sizes and kurtosis and showing
that pooling multiple sites introduces scanner heterogeneity.

This is a runnable tool. It prints REAL numbers computed from the data you
download; nothing is hard-coded. Run it and keep the printed table as evidence.
Requires internet (nilearn downloads ABIDE). Run locally, not in a sandbox.
============================================================

In [ ]:
import sys, numpy as np
sys.path.insert(0, "..")                 # so "src" is importable (run from notebooks/)
from nilearn import datasets
from src.LCT import _kappa_hat, _zscore_columns

In [ ]:
# A spread of ABIDE acquisition sites to compare before selecting one.
SITES = ["NYU", "UM_1", "USM", "UCLA_1", "LEUVEN_2", "PITT", "YALE", "TRINITY", "OHSU", "CALTECH"]
ATLAS = "rois_cc200"                      # Craddock-200, same as the final analysis

In [ ]:
def subject_kappa(ts):
    ts = np.asarray(ts, dtype=float)
    if ts.ndim != 2 or ts.shape[0] < 10 or ts.shape[1] < 2:
        return None
    # drop dead (zero-variance) ROIs before measuring kurtosis
    keep = ts.std(axis=0) > 0
    if keep.sum() < 2:
        return None
    return _kappa_hat(_zscore_columns(ts[:, keep]))

In [ ]:
print(f"{'site':10s} {'n_subj':>6s} {'ASD':>4s} {'CTL':>4s} {'kappa_hat':>10s} {'%>1.1':>6s}")
rows = []
for site in SITES:
    try:
        d = datasets.fetch_abide_pcp(
            SITE_ID=[site], pipeline="cpac",
            band_pass_filtering=True, global_signal_regression=False,
            derivatives=[ATLAS], quality_checked=True, verbose=0)
        ts_list = d[ATLAS]
        ks = [k for k in (subject_kappa(t) for t in ts_list) if k is not None]
        dx = np.asarray(d.phenotypic["DX_GROUP"])
        n_asd, n_ctl = int((dx == 1).sum()), int((dx == 2).sum())
        kmean = float(np.mean(ks)) if ks else float("nan")
        frac = float(np.mean(np.array(ks) > 1.1)) if ks else float("nan")
        print(f"{site:10s} {len(ts_list):6d} {n_asd:4d} {n_ctl:4d} {kmean:10.3f} {frac:6.2f}")
        rows.append((site, len(ts_list), kmean))
    except Exception as e:
        print(f"{site:10s} error: {e}")

In [ ]:
# --- PASTE BACK / KEEP: this table is your site-selection evidence ---
# Reading: sites with the largest n and kappa_hat closest to 1 are the best
# near-Gaussian candidates. NYU was selected on this basis. Pooling all sites
# would mix scanners/protocols, which can look like a group difference.